In [ ]:
import sys

print(sys.executable)

In [ ]:
from pathlib import Path

import openeo
import xarray as xr
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

print("openEO version:", openeo.client_version())

In [ ]:
project_root = Path.cwd()

# Utile nel caso in cui VS Code esegua il notebook
# usando notebooks/ come cartella corrente.
if project_root.name == "notebooks":
    project_root = project_root.parent

raw_data_dir = project_root / "data" / "raw"
raw_data_dir.mkdir(parents=True, exist_ok=True)

output_path = raw_data_dir / "sentinel5p_ch4_test_june_2024.nc"

print("Project root:", project_root)
print("Output file:", output_path)

In [ ]:
connection = openeo.connect(
    url="openeo.dataspace.copernicus.eu"
)

connection.authenticate_oidc()

In [ ]:
collection = connection.describe_collection("SENTINEL_5P_L2")
bands = collection["cube:dimensions"]["bands"]["values"]
assert "CH4" in bands
print("CH4 band available.")

In [ ]:
test_extent = {
    "west": 8.5,
    "south": 44.8,
    "east": 10.5,
    "north": 46.0,
    "crs": "EPSG:4326",
}

test_period = ["2024-06-01", "2024-07-01"]

ch4_cube = connection.load_collection(
    "SENTINEL_5P_L2",
    spatial_extent=test_extent,
    temporal_extent=test_period,
    bands=["CH4"],
)

ch4_cube

In [ ]:
ch4_cube.print_json()

In [ ]:
if not output_path.exists():
    ch4_cube.download(
        outputfile=output_path,
        format="NetCDF",
    )
    print("Download completed.")
else:
    print("File already exists:", output_path)

In [ ]:
ds = xr.open_dataset(output_path)

ds

In [ ]:
print("Dimensions:")
print(ds.sizes)

print("\nCoordinates:")
print(list(ds.coords))

print("\nData variables:")
print(list(ds.data_vars))

In [ ]:
assert "CH4" in ds.data_vars, (
    f"CH4 not found. Available variables: {list(ds.data_vars)}"
)

ch4 = ds["CH4"]

ch4

In [ ]:
print("CH4 dimensions:", ch4.dims)
print("CH4 shape:", ch4.shape)
print("CH4 attributes:", ch4.attrs)

In [ ]:
print("Valid observations:", ch4.count().values)
print("Minimum:", ch4.min(skipna=True).values)
print("Maximum:", ch4.max(skipna=True).values)
print("Mean:", ch4.mean(skipna=True).values)

In [ ]:
time_dim = next(
    (dim for dim in ["t", "time"] if dim in ch4.dims),
    None,
)

print("Time dimension:", time_dim)

In [ ]:
if time_dim is not None:
    first_time = pd.to_datetime(ch4[time_dim].values[0])

    first_observation = ch4.isel({time_dim: 0})

    first_observation.plot(
        figsize=(9, 6),
        robust=True,
    )

    plt.title(f"Sentinel-5P CH₄ — {first_time:%Y-%m-%d}")
    plt.show()
else:
    print("No temporal dimension found.")

In [ ]:
if time_dim is not None:
    spatial_dims = [
        dim for dim in ch4.dims
        if dim != time_dim
    ]

    regional_mean = ch4.mean(
        dim=spatial_dims,
        skipna=True,
    )

    regional_mean.plot(
        marker="o",
        figsize=(10, 5),
    )

    plt.title("Mean satellite-observed CH₄ over the test area")
    plt.ylabel("CH₄")
    plt.grid()
    plt.show()